In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")

In [0]:
from pyspark.sql import functions as F

CATALOG = "cpx_dataanalytics_gold"


# Rolling 365-day window as YYYYMMDD int
window_start = (
    F.date_format(F.date_sub(F.current_date(), 365), "yyyyMMdd").cast("int")
)

# Active CineClub members from latest snapshot
snapshot = spark.table(f"{CATALOG}.marketing.mkt_subscriber_profile_snapshot")
latest_snapshot_date = snapshot.agg(F.max("snapshot_date_id")).first()[0]

active_members = (
    snapshot
    .filter(F.col("snapshot_date_id") == latest_snapshot_date)
    .filter(F.col("renewal_status").like("Active%"))
    .select(F.col("cde_id").cast("bigint").alias("cde_id"))
    .distinct()
)

candy_classes = ["VHO-CANDY", "CANDY", "SMALL CANDY"]

candy_items = (
    spark.table(f"{CATALOG}.customer360.cpx_d_item")
    .filter(F.col("Item_Class_Description1").isin(candy_classes))
    .filter(~F.col("Item_Description").like("DELIVERY%"))
    .filter(~F.col("Item_Description").like("VHO DELIVERY%"))
    .filter(~F.col("Item_Description").like("SNACK%"))
    .select("itemskey", "Item_Description")
)

fact = spark.table(f"{CATALOG}.customer360.c360_f_transaction")

candy_transactions = (
    fact
    .filter(F.col("VisitDateId") >= window_start)
    .filter(F.col("ConcessionFLAG") == "Yes")
    # .filter(F.col("CineClubFLAG") == "Yes")
    .join(active_members, fact["CDE_ID"] == active_members["cde_id"], "inner")
    .join(candy_items, fact["ItemSkey"] == candy_items["itemskey"], "inner")
    .select(
        candy_items["Item_Description"],
        fact["CDE_ID"],
        fact["Quantity"],
        fact["Net_Revenue"],
    )
)

# Final aggregate
candy_sales = (
    candy_transactions
    .withColumn(
        "Size",
        F.when(F.col("Item_Description").like("% SUP %"), "Large").otherwise("Regular")
    )
    .groupBy("Item_Description", "Size")
    .agg(
        F.sum("Quantity").alias("units_sold"),
        F.count("*").alias("member_purchase_count"),
        F.sum("Net_Revenue").alias("net_revenue"),
    )
    .orderBy(F.col("units_sold").desc())
)

candy_sales.display()

In [0]:
from pyspark.sql.window import Window

modal_price_window = (
    Window.partitionBy("Item_Description")
          .orderBy(F.col("price_freq").desc(), F.col("Net_Revenue").desc())
)

candy_sales = (
    candy_transactions
    .withColumn(
        "Size",
        F.when(F.col("Item_Description").like("% SUP %"), "Large").otherwise("Regular")
    )
    .groupBy("Item_Description", "Size")
    .agg(
        F.sum("Quantity").alias("units_sold"),
        F.count("*").alias("member_purchase_count"),
        F.sum("Net_Revenue").alias("net_revenue"),
        F.round(F.sum("Net_Revenue") / F.sum("Quantity"), 2).alias("avg_price"),
    )
    .join(
        candy_transactions
            .filter(F.col("Quantity") == 1)
            .groupBy("Item_Description", "Net_Revenue")
            .agg(F.count("*").alias("price_freq"))
            .withColumn("rn", F.row_number().over(modal_price_window))
            .filter(F.col("rn") == 1)
            .select("Item_Description", F.col("Net_Revenue").alias("modal_price")),
        on="Item_Description",
        how="left"
    )
    .orderBy(F.col("units_sold").desc())
)

candy_sales.display()

In [0]:
# All-candy line items (no CC member or CineClub filter) — used for totals and modal price
all_candy_lines = (
    fact
    .filter(F.col("VisitDateId") >= window_start)
    .filter(F.col("ConcessionFLAG") == "Yes")
    .join(candy_items, fact["ItemSkey"] == candy_items["itemskey"], "inner")
    .select(
        candy_items["Item_Description"],
        fact["Quantity"],
        fact["Net_Revenue"],
    )
)

all_candy_totals = (
    all_candy_lines
    .groupBy("Item_Description")
    .agg(
        F.sum("Quantity").alias("total_units_sold"),
        F.count("*").alias("total_purchase_count"),
        F.sum("Net_Revenue").alias("total_net_revenue"),
    )
)

candy_sales = (
    candy_transactions
    .withColumn(
        "Size",
        F.when(F.col("Item_Description").like("% SUP %"), "Large").otherwise("Regular")
    )
    .groupBy("Item_Description", "Size")
    .agg(
        F.sum("Quantity").alias("cc_units_sold"),
        F.count("*").alias("cc_purchase_count"),
        F.sum("Net_Revenue").alias("cc_net_revenue"),
        F.round(F.sum("Net_Revenue") / F.sum("Quantity"), 2).alias("avg_price"),
    )
    .join(
        all_candy_lines  # ← modal price now from all transactions
            .filter(F.col("Quantity") == 1)
            .groupBy("Item_Description", "Net_Revenue")
            .agg(F.count("*").alias("price_freq"))
            .withColumn("rn", F.row_number().over(modal_price_window))
            .filter(F.col("rn") == 1)
            .select("Item_Description", F.col("Net_Revenue").alias("modal_price")),
        on="Item_Description",
        how="left"
    )
    .join(all_candy_totals, on="Item_Description", how="left")
    .withColumn(
        "cc_share_of_units",
        F.round(F.col("cc_units_sold") / F.col("total_units_sold"), 4)
    )
    .orderBy(F.col("total_units_sold").desc())
)

candy_sales.display()

In [0]:

theatre_locations = (
    spark.table(f"{CATALOG}.customer360.cpx_d_location")
    .filter(F.col("LC_Location_Type_Description") == "Cineplex Theatre")
    .select(F.col("LocationID").alias("loc_id"))
)
# Count candy units per active member (CC-attributed, theatre, last 365d)
member_candy_counts = (
    fact
    .filter(F.col("VisitDateId") >= window_start)
    .filter(F.col("ConcessionFLAG") == "Yes")
    .filter(F.col("CineClubFLAG") == "Yes")
    .join(theatre_locations, fact["LocationId"] == theatre_locations["loc_id"], "inner")
    .join(active_members, fact["CDE_ID"] == active_members["cde_id"], "inner")
    .join(candy_items, fact["ItemSkey"] == candy_items["itemskey"], "inner")
    .groupBy(active_members["cde_id"])
    .agg(F.sum(fact["Quantity"]).alias("candy_units"))
)

# Left-join back to active_members so zero-candy members are kept and bucketed as 0
member_segments = (
    active_members
    .join(member_candy_counts, on="cde_id", how="left")
    .withColumn("candy_units", F.coalesce(F.col("candy_units"), F.lit(0)))
    .withColumn(
        "candy_segment",
        F.when(F.col("candy_units") == 0, "0 — none")
         .when(F.col("candy_units") <= 2, "1–2 — tried")
         .when(F.col("candy_units") <= 5, "3–5 — occasional")
         .when(F.col("candy_units") <= 10, "6–10 — regular")
         .when(F.col("candy_units") <= 20, "11–20 — frequent")
         .otherwise("21+ — heavy")
    )
    .withColumn(
        "sort_order",
        F.when(F.col("candy_segment") == "0 — none", 0)
         .when(F.col("candy_segment") == "1–2 — tried", 1)
         .when(F.col("candy_segment") == "3–5 — occasional", 2)
         .when(F.col("candy_segment") == "6–10 — regular", 3)
         .when(F.col("candy_segment") == "11–20 — frequent", 4)
         .otherwise(5)
    )
)

# Aggregate by segment
segment_distribution = (
    member_segments
    .groupBy("candy_segment", "sort_order")
    .agg(
        F.count("*").alias("members"),
        F.sum("candy_units").alias("total_units_in_segment"),
    )
    .withColumn(
        "pct_of_members",
        F.round(F.col("members") * 100.0 / F.sum("members").over(Window.partitionBy()), 2)
    )
    .orderBy("sort_order")
    .drop("sort_order")
)

segment_distribution.display()